# 第 3 章：Tokenizer 与 Dataset

这个 notebook 对应 `lessons/03_tokenizer_and_dataset.md`，演示字符级 tokenizer、batch padding、language modeling dataset 和 SFT label mask。

In [ ]:
import torch

from src.data.text_datasets import (
    IGNORE_INDEX,
    ChatMessage,
    LanguageModelingTextDataset,
    build_sft_features,
)
from src.tokenizer.simple_tokenizer import CharacterTokenizer

## 1. Encode / Decode

字符级 tokenizer 的优点是透明：每个字符就是一个 token，特殊 token 固定在词表前 4 个位置。

In [ ]:
text = "中文 tokenizer"
tokenizer = CharacterTokenizer.from_texts([text])

ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))
print(
    tokenizer.pad_token_id,
    tokenizer.unk_token_id,
    tokenizer.bos_token_id,
    tokenizer.eos_token_id,
)

## 2. Batch Padding 与 Attention Mask

`attention_mask=1` 表示真实 token，`attention_mask=0` 表示 padding。后续模型和 loss 都要尊重这个边界。

In [ ]:
batch = tokenizer.batch_encode(["中文", "中文 tokenizer"], max_length=16)
print(batch.input_ids)
print(batch.attention_mask)

## 3. Language Modeling Dataset

LM dataset 的关键是右移：`labels[t]` 是 `input_ids[t]` 的下一个 token。

In [ ]:
token_ids = torch.tensor(tokenizer.encode("中文 tokenizer 中文 tokenizer"), dtype=torch.long)
dataset = LanguageModelingTextDataset(token_ids, block_size=6)
input_ids, labels = dataset[0]
print(input_ids)
print(labels)
print(tokenizer.decode(input_ids), "->", tokenizer.decode(labels))

## 4. SFT Label Mask

SFT 里通常只让 assistant 内容参与 loss。system/user/padding 的 label 应为 `-100`。

In [ ]:
chat_texts = [
    "<|system|>\n"
    "你是助教\n"
    "<|user|>\n"
    "解释 causal mask\n"
    "<|assistant|>\n"
    "只能看历史 token\n"
]
chat_tokenizer = CharacterTokenizer.from_texts(chat_texts)
features = build_sft_features(
    [
        ChatMessage(role="system", content="你是助教"),
        ChatMessage(role="user", content="解释 causal mask"),
        ChatMessage(role="assistant", content="只能看历史 token"),
    ],
    tokenizer=chat_tokenizer,
    max_length=80,
)

print(features.input_ids)
print(features.labels)
supervised = features.labels[features.labels != IGNORE_INDEX]
print("supervised text:", chat_tokenizer.decode(supervised))